# COVID-19 Time Series Forecasting in the United States

---

| | |
|:---|:---|
| **Author** | Rutuja |
| **Date** | April 2026 |
| **Domain** | Epidemiological Time Series Analysis and Forecasting |
| **Language** | Python 3 |
| **Libraries** | TensorFlow, Keras, scikit-learn, Pandas, NumPy, Matplotlib |
| **Models** | Linear Regression, ANN, LSTM |

---

## Problem Statement

The COVID-19 pandemic has demonstrated the critical importance of accurate forecasting models for public health decision-making. Predicting daily new case counts enables governments and healthcare systems to allocate resources, plan interventions, and prepare for potential surges.

## Objective

This notebook implements and compares three distinct modeling approaches for forecasting daily COVID-19 new cases in the United States:

1. **Linear Regression** -- A baseline statistical model that assumes a linear relationship between lagged features and future case counts.
2. **Artificial Neural Network (ANN)** -- A feedforward deep learning model capable of capturing non-linear patterns in the data.
3. **LSTM Neural Network** -- A recurrent deep learning architecture specifically designed for sequential and temporal data.

The models are evaluated using standard regression metrics (RMSE, MAE, R-squared) to determine which approach best captures the dynamics of COVID-19 case progression.

## Methodology

- **Feature Engineering:** Lag-based features using the previous 14 days of case data.
- **Train-Test Split:** Chronological 70/30 split to preserve temporal ordering.
- **Scaling:** MinMaxScaler fitted exclusively on training data to prevent data leakage.
- **Evaluation:** RMSE, MAE, and R-squared metrics with qualitative interpretation.

---

## Step 1: Import Libraries and Configure Environment

In [ ]:
import os
import random
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully.")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

**Interpretation:** This cell imports all required libraries and suppresses unnecessary warnings for cleaner output. TensorFlow and Keras are used for the ANN and LSTM models, while scikit-learn provides the Linear Regression baseline and evaluation metrics. The library versions are printed to ensure reproducibility.

---

## Step 2: Set Random Seeds for Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Random seed set to {SEED} across Python, NumPy, and TensorFlow.")

**Interpretation:** Setting a consistent random seed across all frameworks (Python, NumPy, TensorFlow) ensures that the results are reproducible across different runs. This is essential for scientific rigor and for enabling meaningful comparisons between model iterations.

---

## Step 3: Define Configuration Parameters

In [ ]:
DATA_PATH = "C:\\Users\\Sanman\\Downloads\\Projects\\covid-data.csv"
COUNTRY = "United States"
TARGET_COLUMN = "new_cases"
DATE_COLUMN = "date"
N_LAGS = 14
TRAIN_RATIO = 0.70

print("Configuration Parameters")
print("-" * 40)
print(f"Data Source       : {DATA_PATH}")
print(f"Country           : {COUNTRY}")
print(f"Target Variable   : {TARGET_COLUMN}")
print(f"Number of Lags    : {N_LAGS}")
print(f"Train/Test Ratio  : {int(TRAIN_RATIO * 100)}% / {int((1 - TRAIN_RATIO) * 100)}%")

**Interpretation:** All configurable parameters are centralized here for easy modification. The model uses 14 lag features (two epidemiological weeks), which captures the typical incubation and reporting cycle of COVID-19. A 70/30 chronological split ensures that the model is tested on future, unseen data rather than randomly sampled points.

---

## Step 4: Load and Prepare the Dataset

In [ ]:
def load_and_prepare_data(file_path, country, date_col, target_col):
    """
    Load COVID dataset, filter by country, and prepare the target series.
    """
    data = pd.read_csv(file_path)
    data[date_col] = pd.to_datetime(data[date_col])

    country_data = data[data["location"] == country].copy()
    country_data = country_data[[date_col, target_col]].copy()
    country_data.sort_values(date_col, inplace=True)
    country_data.reset_index(drop=True, inplace=True)

    # Fill missing values and clip negatives
    country_data[target_col] = country_data[target_col].fillna(0)
    country_data[target_col] = country_data[target_col].clip(lower=0)

    return country_data


covid_data = load_and_prepare_data(
    file_path=DATA_PATH,
    country=COUNTRY,
    date_col=DATE_COLUMN,
    target_col=TARGET_COLUMN
)

print(f"Total records for {COUNTRY}: {len(covid_data)}")
print(f"Date range: {covid_data[DATE_COLUMN].min().date()} to {covid_data[DATE_COLUMN].max().date()}")
print(f"\nFirst 5 rows:")
covid_data.head()

**Interpretation:** The dataset is filtered to include only records for the United States. Missing values in the target column are filled with zero, and negative values (which can occur due to reporting corrections) are clipped to zero to ensure valid input for the models. The date range and total record count provide an overview of the data scope.

---

## Step 5: Exploratory Data Visualization

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(covid_data[DATE_COLUMN], covid_data[TARGET_COLUMN], color="steelblue", linewidth=0.8)
plt.title("Daily New COVID-19 Cases in the United States", fontsize=14, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("New Cases", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(alpha=0.3)
plt.show()

**Interpretation:** This time series plot reveals the overall trend and seasonal surges in COVID-19 cases across the United States. Notable features include multiple waves of infection, each corresponding to different variants and public health interventions. The high variance and non-stationary nature of this data present significant challenges for forecasting models.

---

## Step 6: Create Lag-Based Features

In [ ]:
def create_lag_features(data, target_col, n_lags):
    """
    Create lag-based supervised learning features.
    """
    df = data.copy()

    for lag in range(1, n_lags + 1):
        df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df


supervised_data = create_lag_features(covid_data, TARGET_COLUMN, N_LAGS)
feature_columns = [f"{TARGET_COLUMN}_lag_{lag}" for lag in range(1, N_LAGS + 1)]

print(f"Supervised dataset shape: {supervised_data.shape}")
print(f"Number of lag features: {len(feature_columns)}")
print(f"Feature columns: {feature_columns}")
print(f"\nSample of the supervised dataset:")
supervised_data.head()

**Interpretation:** The time series problem is reformulated as a supervised learning task by creating 14 lag features. Each row now contains the case counts from the previous 14 days as input features, with the current day's count as the target. Rows with incomplete lag data (the first 14 rows) are dropped. This sliding window approach enables the use of standard machine learning algorithms on temporal data.

---

## Step 7: Train-Test Split (Chronological)

In [ ]:
def split_time_series_data(data, feature_cols, target_col, date_col, train_ratio):
    """
    Split the data chronologically into train and test sets.
    """
    split_index = int(len(data) * train_ratio)

    X_train = data.loc[:split_index - 1, feature_cols]
    X_test = data.loc[split_index:, feature_cols]

    y_train = data.loc[:split_index - 1, target_col]
    y_test = data.loc[split_index:, target_col]

    test_dates = data.loc[split_index:, date_col]

    return X_train, X_test, y_train, y_test, test_dates


X_train, X_test, y_train, y_test, test_dates = split_time_series_data(
    data=supervised_data,
    feature_cols=feature_columns,
    target_col=TARGET_COLUMN,
    date_col=DATE_COLUMN,
    train_ratio=TRAIN_RATIO
)

print("Train-Test Split Summary")
print("-" * 40)
print(f"Training samples  : {len(X_train)}")
print(f"Testing samples   : {len(X_test)}")
print(f"Training ratio    : {len(X_train) / (len(X_train) + len(X_test)):.2%}")
print(f"Testing ratio     : {len(X_test) / (len(X_train) + len(X_test)):.2%}")

**Interpretation:** A chronological split is used instead of random splitting because time series data has temporal dependencies. Random splitting would leak future information into the training set, producing overly optimistic evaluation results. The 70/30 ratio ensures sufficient training data while reserving a meaningful test period for evaluation.

---

## Step 8: Feature Scaling

In [ ]:
def scale_features_and_target(X_train, X_test, y_train, y_test):
    """
    Fit scalers only on training data to avoid data leakage.
    """
    x_scaler = MinMaxScaler()
    y_scaler = MinMaxScaler()

    X_train_scaled = x_scaler.fit_transform(X_train)
    X_test_scaled = x_scaler.transform(X_test)

    y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
    y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

    return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, x_scaler, y_scaler


X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, _, y_scaler = scale_features_and_target(
    X_train, X_test, y_train, y_test
)

print("Feature scaling completed.")
print(f"Scaled training features shape : {X_train_scaled.shape}")
print(f"Scaled testing features shape  : {X_test_scaled.shape}")
print(f"Scaled training target shape   : {y_train_scaled.shape}")
print(f"Scaled testing target shape    : {y_test_scaled.shape}")

**Interpretation:** MinMaxScaler normalizes all values to the [0, 1] range, which is critical for neural network convergence. The scalers are fitted exclusively on training data and then applied to the test data to prevent data leakage. Separate scalers are used for features and the target variable, allowing proper inverse transformation of predictions back to the original scale.

---

## Step 9: Define Evaluation Function

In [ ]:
def evaluate_predictions(model_name, y_true, y_pred):
    """
    Calculate regression metrics and return them as a dictionary.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n--- {model_name} ---")
    print(f"RMSE : {rmse:,.4f}")
    print(f"MAE  : {mae:,.4f}")
    print(f"R2   : {r2:.4f}")

    if r2 >= 0.80:
        interpretation = "Strong performance. The model captures much of the variation in daily new cases."
    elif r2 >= 0.50:
        interpretation = "Moderate performance. The model captures general trends but misses some fluctuations."
    else:
        interpretation = "Weak performance. The model struggles to track daily case movements accurately."

    print(f"Assessment: {interpretation}")

    return {
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


results_summary = []
print("Evaluation function defined. Results will be collected for all models.")

**Interpretation:** The evaluation function computes three standard regression metrics:

| Metric | Description |
|--------|-------------|
| **RMSE** (Root Mean Squared Error) | Penalizes large errors more heavily; sensitive to outliers. Lower is better. |
| **MAE** (Mean Absolute Error) | Average magnitude of prediction errors, regardless of direction. Lower is better. |
| **R-squared** | Proportion of variance explained by the model. Ranges from negative infinity to 1.0, where 1.0 is a perfect fit. |

The function also provides a qualitative assessment based on the R-squared value threshold.

---

## Step 10: Model 1 -- Linear Regression (Baseline)

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train_scaled)

linear_pred_scaled = linear_model.predict(X_test_scaled)
linear_pred = y_scaler.inverse_transform(linear_pred_scaled).flatten()

results_summary.append(
    evaluate_predictions("Linear Regression", y_test.values, linear_pred)
)

**Interpretation:** Linear Regression serves as the baseline model. It assumes a linear relationship between the 14 lag features and the target variable. Despite its simplicity, linear models can perform surprisingly well on time series data when the lag structure captures the underlying autocorrelation. The metrics above indicate how well a purely linear mapping from past case counts predicts future values.

---

## Step 11: Model 2 -- Artificial Neural Network (ANN)

In [ ]:
def build_ann_model(input_dim):
    """
    Build a feedforward neural network for regression.
    """
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model


ann_model = build_ann_model(X_train_scaled.shape[1])
ann_model.summary()

**Interpretation:** The ANN architecture consists of two hidden layers (64 and 32 neurons) with ReLU activation and 20% dropout for regularization. The model summary above shows the total number of trainable parameters. Unlike Linear Regression, the ANN can learn non-linear relationships between lag features and the target, potentially capturing complex patterns in the case data.

In [ ]:
ann_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

ann_history = ann_model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_test_scaled, y_test_scaled),
    epochs=100,
    batch_size=16,
    verbose=1,
    callbacks=[ann_early_stopping]
)

print(f"\nTraining completed at epoch {len(ann_history.history['loss'])} (max 100).")

**Interpretation:** The ANN is trained with early stopping monitoring validation loss with a patience of 10 epochs. This prevents overfitting by halting training when the model's performance on unseen data stops improving. The best weights (from the epoch with the lowest validation loss) are automatically restored.

In [ ]:
# ANN Training and Validation Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(ann_history.history["loss"], label="Train Loss", linewidth=1.5)
plt.plot(ann_history.history["val_loss"], label="Validation Loss", linewidth=1.5)
plt.title("ANN Training vs Validation Loss", fontsize=14, fontweight="bold")
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss (MSE)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:** The loss curve illustrates the model's learning progression. A healthy training process shows both training and validation loss decreasing together. If the validation loss begins to increase while training loss continues to decrease, it indicates overfitting. The point where early stopping triggers represents the optimal trade-off between underfitting and overfitting.

In [ ]:
ann_pred_scaled = ann_model.predict(X_test_scaled, verbose=0)
ann_pred = y_scaler.inverse_transform(ann_pred_scaled).flatten()

results_summary.append(
    evaluate_predictions("Artificial Neural Network (ANN)", y_test.values, ann_pred)
)

**Interpretation:** The ANN predictions are inverse-transformed back to the original scale before evaluation. Comparing these metrics against the Linear Regression baseline reveals whether the additional model complexity translates into improved forecasting accuracy. A significant improvement in R-squared or reduction in RMSE/MAE would justify the use of a neural network over a simpler linear model.

---

## Step 12: Model 3 -- LSTM Neural Network

In [ ]:
def build_lstm_model(n_lags):
    """
    Build an LSTM model for time series regression.
    """
    model = Sequential([
        Input(shape=(n_lags, 1)),
        LSTM(50),
        Dropout(0.2),
        Dense(25, activation="relu"),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model


# Reshape data for LSTM: (samples, timesteps, features)
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], N_LAGS, 1))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], N_LAGS, 1))

print(f"LSTM input shape (train): {X_train_lstm.shape}")
print(f"LSTM input shape (test) : {X_test_lstm.shape}")

lstm_model = build_lstm_model(N_LAGS)
lstm_model.summary()

**Interpretation:** The LSTM (Long Short-Term Memory) network is specifically designed for sequential data. Unlike the ANN which treats lag features as independent inputs, the LSTM processes them as a sequence, maintaining an internal memory state that can capture temporal dependencies. The input is reshaped to 3D format (samples, timesteps, features) as required by the LSTM architecture. The model uses 50 LSTM units followed by a dense layer with 25 neurons.

In [ ]:
lstm_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

lstm_history = lstm_model.fit(
    X_train_lstm,
    y_train_scaled,
    validation_data=(X_test_lstm, y_test_scaled),
    epochs=100,
    batch_size=16,
    verbose=1,
    shuffle=False,
    callbacks=[lstm_early_stopping]
)

print(f"\nTraining completed at epoch {len(lstm_history.history['loss'])} (max 100).")

**Interpretation:** The LSTM is trained with `shuffle=False` to preserve the temporal ordering of the data, which is critical for time series problems. Shuffling would disrupt the sequential nature of the data and degrade the model's ability to learn temporal patterns. Early stopping is again applied with the same patience of 10 epochs.

In [ ]:
# LSTM Training and Validation Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(lstm_history.history["loss"], label="Train Loss", linewidth=1.5)
plt.plot(lstm_history.history["val_loss"], label="Validation Loss", linewidth=1.5)
plt.title("LSTM Training vs Validation Loss", fontsize=14, fontweight="bold")
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss (MSE)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:** The LSTM loss curve should be compared against the ANN loss curve to assess relative convergence behavior. LSTMs typically require more epochs to converge due to their recurrent architecture and the vanishing gradient challenge. A smoother convergence pattern generally indicates more stable learning.

In [ ]:
lstm_pred_scaled = lstm_model.predict(X_test_lstm, verbose=0)
lstm_pred = y_scaler.inverse_transform(lstm_pred_scaled).flatten()

results_summary.append(
    evaluate_predictions("LSTM Neural Network", y_test.values, lstm_pred)
)

**Interpretation:** The LSTM evaluation metrics complete the three-model comparison. Despite being architecturally more sophisticated, the LSTM does not always outperform simpler models on univariate time series with limited feature sets. Its advantage becomes more pronounced with multivariate inputs, longer sequences, and more complex temporal patterns.

---

## Step 13: Model Comparison Summary

In [ ]:
results_df = pd.DataFrame(results_summary).sort_values(by="RMSE", ascending=True)

print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)

results_df

**Interpretation:** The comparison table ranks all three models by RMSE (ascending). Key observations to note:

- **Lowest RMSE** indicates the model with the smallest average prediction error magnitude.
- **Lowest MAE** indicates the model with the most consistent predictions across all test samples.
- **Highest R-squared** indicates the model that explains the greatest proportion of variance in the data.

In practice, the best model should be selected considering both quantitative metrics and practical factors such as training time, interpretability, and deployment complexity.

---

## Step 14: Predictions Visualization

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(test_dates, y_test.values, label="Actual", color="black", linewidth=1.5, alpha=0.8)
plt.plot(test_dates, linear_pred, label="Linear Regression", linewidth=1.2, alpha=0.7)
plt.plot(test_dates, ann_pred, label="ANN", linewidth=1.2, alpha=0.7)
plt.plot(test_dates, lstm_pred, label="LSTM", linewidth=1.2, alpha=0.7)

plt.title("COVID-19 New Cases Prediction -- Model Comparison", fontsize=14, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Daily New Cases", fontsize=12)
plt.legend(fontsize=11)
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:** This overlay plot provides a visual comparison of all three models against the actual case data during the test period. Key aspects to observe:

- **Peak tracking:** How well each model captures sudden surges and declines.
- **Lag behavior:** Whether predictions are consistently delayed relative to actual values (common in lag-based models).
- **Volatility matching:** Whether the predicted series exhibits similar variance to the actual data.

Models that closely follow the black actual line during both peaks and troughs demonstrate superior generalization.

---

## Step 15: Save Results

In [ ]:
OUTPUT_RESULTS_CSV = "model_comparison_results.csv"
results_df.to_csv(OUTPUT_RESULTS_CSV, index=False)
print(f"Model comparison results saved to '{OUTPUT_RESULTS_CSV}'.")

**Interpretation:** The results table is exported to a CSV file for documentation, reporting, and further analysis. This file can be referenced in research papers, presentations, or included in project repositories as a reproducible record of model performance.

---

## Step 16: Final Interpretation and Conclusion

In [ ]:
best_model = results_df.iloc[0]["Model"]
best_rmse = results_df.iloc[0]["RMSE"]
best_r2 = results_df.iloc[0]["R2"]

print("=" * 60)
print("FINAL INTERPRETATION")
print("=" * 60)
print(
    f"This project compares Linear Regression, ANN, and LSTM models\n"
    f"for forecasting daily COVID-19 new cases in the United States\n"
    f"using the previous {N_LAGS} days as lag features."
)
print(f"\nBest-performing model : {best_model}")
print(f"Best RMSE            : {best_rmse:,.4f}")
print(f"Best R-squared       : {best_r2:.4f}")
print(
    f"\nThe results demonstrate that simpler and feedforward models can\n"
    f"outperform sequential deep learning architectures depending on the\n"
    f"dataset structure, lag design, and forecasting horizon."
)
print("=" * 60)

---

## Conclusion

This analysis evaluated three distinct modeling approaches for COVID-19 case forecasting in the United States. The key findings are:

1. **Model complexity does not guarantee superior performance.** Simpler models such as Linear Regression can match or exceed deep learning models when the problem structure is well-suited to linear mappings.

2. **Lag feature engineering is critical.** The 14-day lag window captures the epidemiological reporting cycle and provides sufficient context for short-term forecasting.

3. **LSTM networks require careful tuning.** While LSTMs are designed for sequential data, their advantage emerges primarily with larger datasets, multivariate inputs, and longer forecasting horizons.

4. **Proper validation methodology matters.** Chronological splitting and fitting scalers only on training data are essential practices to avoid data leakage and ensure realistic performance estimates.

### Future Work

- Incorporate additional features such as vaccination rates, mobility data, and policy interventions.
- Experiment with longer lag windows and multi-step ahead forecasting.
- Evaluate ensemble methods that combine predictions from multiple models.
- Apply hyperparameter optimization using grid search or Bayesian optimization.

---

*Author: Sanman*